# TensorCore-FPGA Demo

This notebook demonstrates how to use the TensorCore LLM accelerator on PYNQ-Z1.

## Prerequisites
- Upload `tensorcore.bit` and `tensorcore.hwh` to the same directory as this notebook
- Install the driver: `pip install -e /path/to/pynq/`

In [ ]:
# Import libraries
from pynq import Overlay
import numpy as np
import time

In [ ]:
# Load the overlay
overlay = Overlay("tensorcore.bit")
print("Overlay loaded successfully!")
overlay?

In [ ]:
# Import TensorCore driver
from tensorcore import TensorCoreDriver

# Initialize driver
tc = TensorCoreDriver(overlay)
print(tc)

## Basic Register Access

In [ ]:
# Check status
print(f"Is Done: {tc.is_done}")
print(f"Is Busy: {tc.is_busy}")
print(f"State: {tc.state}")

In [ ]:
# Configure for 4x4 matrix multiply
tc.configure(k1=4, k2=4, k3=4)
print("Configured for 4x4 x 4x4 matrix multiply")

## Run Computation

In [ ]:
# Start computation (without VPU)
start_time = time.time()
tc.start(vpu_enable=False)

# Wait for completion
success = tc.wait_done(timeout_ms=1000)
elapsed = time.time() - start_time

if success:
    print(f"Computation complete in {elapsed*1000:.2f} ms")
else:
    print("Timeout!")

In [ ]:
# Get performance counters
perf = tc.get_performance()
print(f"Performance Counters:")
print(f"  Cycles:      {perf['cycles']}")
print(f"  Mem Reads:   {perf['mem_reads']}")
print(f"  Mem Writes:  {perf['mem_writes']}")

## VPU Operations

In [ ]:
# Run with VPU L2 Normalization
tc.reset()
tc.configure(k1=4, k2=4, k3=4)
tc.set_vpu_operation(tc.VPU_OP_NORM)

tc.start(vpu_enable=True)
tc.wait_done()

print(f"VPU Norm completed in {tc.perf_cycles} cycles")

In [ ]:
# Run with VPU Softmax
tc.reset()
tc.configure(k1=4, k2=4, k3=4)
tc.set_vpu_operation(tc.VPU_OP_SOFTMAX)

tc.start(vpu_enable=True)
tc.wait_done()

print(f"VPU Softmax completed in {tc.perf_cycles} cycles")

## Cleanup

In [ ]:
# Reset accelerator
tc.reset()
print("TensorCore reset complete")